# Lektion 10 - AI-agenter i produktion

I denne lektion vil du lære **produktionsmønstre** for AI-agenter ved hjælp af Microsoft Agent Framework (Python). Vi dækker:

- **Observabilitet** — tilføjelse af tidsmåling og logning til agentinteraktioner
- **Evaluering** — brug af en evalueringsagent til at bedømme svarenes kvalitet
- **Omkostningsstyring** — strategier for tokenoptimering og modelvalg

Scenariet er en **rejseagent**, der hjælper brugere med at planlægge rejser, med overvågning og evaluering lagdelt ovenpå.


## Opsætning


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Produktionsovervejelser

At flytte AI-agenter fra prototyper til produktion kræver omhyggelig opmærksomhed på tre søjler:

1. **Observabilitet** — Du har brug for indsigt i, hvad agenten gør, hvor lang tid det tager, og hvilke værktøjer den kalder. Uden sporing og logning er det næsten umuligt at fejlfinde problemer i produktionen.

2. **Evaluering** — Automatiserede kvalitetskontroller sikrer, at agentens svar forbliver nøjagtige, fyldestgørende og nyttige over tid. En evalueringsagent kan score svar i forhold til definerede kriterier.

3. **Omkostningsstyring** — Forbrug af tokens påvirker direkte omkostningerne. Strategier som promptoptimering, modelvalg og caching hjælper med at holde udgifterne under kontrol uden at gå på kompromis med kvaliteten.


## Opbygning af en observerbar agent

Vi definerer rejseværktøjer og pakker agentkaldet ind med timing, så vi kan overvåge latenstid. I produktion ville du integrere med OpenTelemetry eller en lignende tracing-backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evalueringsmønstre

Et almindeligt produktionsmønster er at bruge en anden agent som en **evaluatør**. Evaluatøren scorer den primære agents svar i forhold til foruddefinerede kriterier såsom fuldstændighed, nøjagtighed og hjælpsomhed.

Dette muliggør:
- Automatiserede kvalitetskontroller, før svar når brugerne
- Registrering af regressioner, når prompts eller modeller ændres
- Kontinuerlig overvågning af agentens præstation over tid


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategier for omkostningsstyring

At kontrollere omkostninger er afgørende for produktions-AI-agenter. Her er nøglestrategier:

| Strategy | Description |
|---|---|
| **Prompt optimization** | Hold systeminstruktioner korte. Fjern redundant kontekst for at reducere antallet af inputtokens. |
| **Model selection** | Brug mindre, billigere modeller (f.eks. GPT-4o-mini) til simple opgaver som klassificering eller ekstraktion, og reserver større modeller til komplekst ræsonnement. |
| **Caching** | Cache værktøjsresultater og hyppige forespørgsler for at undgå redundante API-opkald. |
| **Token budgets** | Sæt `max_tokens`-grænser for at forhindre uventet lange svar. |
| **Batching** | Saml flere brugerforespørgsler i ét API-opkald, hvor det er muligt. |

I praksis fungerer en lagdelt tilgang godt: diriger ligetil anmodninger til en hurtig, billig model, og eskaler kun komplekse forespørgsler til en mere kapabel.


## Resumé

I denne lektion lærte du, hvordan du:

1. **Tilføj observabilitet** til agentinteraktioner med tidsmåling og logning, hvilket lægger grundlaget for sporing og overvågning.
2. **Evaluer agenters svar** automatisk ved hjælp af en evalueringsagent, der vurderer fuldstændighed, nøjagtighed og hjælpsomhed.
3. **Håndter omkostninger** gennem promptoptimering, modelvalg, caching og tokenbudgetter.

Disse produktionsmønstre hjælper med at sikre, at dine AI-agenter er pålidelige, målbare og omkostningseffektive i stor skala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Ansvarsfraskrivelse:
Dette dokument er oversat ved hjælp af AI-oversættelsestjenesten Co-op Translator (https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, bedes du være opmærksom på, at automatiske oversættelser kan indeholde fejl eller unøjagtigheder. Det oprindelige dokument på originalsproget bør betragtes som den autoritative kilde. For kritisk information anbefales en professionel, menneskelig oversættelse. Vi er ikke ansvarlige for eventuelle misforståelser eller fejltolkninger, som måtte opstå som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
